# M4 Performance Analysis — Distributed Multi-GPU LBVH

This notebook parses timing output from `lbvh_mpi` scaling runs and produces:
- Strong-scaling speedup and parallel efficiency curves
- Weak-scaling efficiency curves
- Per-stage breakdown (compute vs. communication)
- Isoefficiency analysis
- Roofline comparison with M3 single-GPU numbers

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Parse timing output

Paste or load the stdout from `sbatch/logs/strong_*.out` and `weak_*.out` below.

`lbvh_mpi` prints a table like:
```
Stage                  min(ms)   mean(ms)   max(ms)
A  Morton encode       ...       ...        ...
B1 local CUB sort      ...       ...        ...
B2 Allgather           ...       ...        ...
B3 Alltoallv           ...       ...        ...
C  Karras+refit        ...       ...        ...
D  top-level tree      ...       ...        ...
   TOTAL               ...       ...        ...
```

In [ ]:
STAGE_NAMES = ['A_morton', 'B1_sort', 'B2_allgather', 'B3_alltoallv', 'C_build', 'D_merge', 'total']

def parse_lbvh_mpi_output(text):
    """Parse one block of lbvh_mpi stdout. Returns dict stage->mean_ms."""
    result = {}
    lines = text.strip().splitlines()
    stage_idx = 0
    for line in lines:
        m = re.search(r'(\d+\.\d+)\s+(\d+\.\d+)\s+(\d+\.\d+)', line)
        if m and stage_idx < len(STAGE_NAMES):
            result[STAGE_NAMES[stage_idx]] = float(m.group(2))  # mean
            stage_idx += 1
    return result


def load_log_file(path):
    """Load log file and return list of (P, timings_dict) parsed from '--- P=N ranks ---' blocks."""
    text = Path(path).read_text()
    runs = []
    blocks = re.split(r'--- P=(\d+) ranks? ---', text)
    for i in range(1, len(blocks), 2):
        P = int(blocks[i])
        body = blocks[i + 1]
        timings = parse_lbvh_mpi_output(body)
        if timings:
            runs.append((P, timings))
    return runs

In [ ]:
from pathlib import Path

LOG_DIR = Path('../sbatch/logs')
OUT_DIR = Path('../latex')
OUT_DIR.mkdir(exist_ok=True)

def _latest(pat):
    files = sorted(LOG_DIR.glob(pat), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(f"No files matching {LOG_DIR / pat}")
    return files[-1]

strong_path = _latest('strong_*.out')
weak_path   = _latest('weak_*.out')
print(f"Strong log : {strong_path}")
print(f"Weak log   : {weak_path}")

STRONG_DATA = {P: t for P, t in load_log_file(strong_path)}
WEAK_DATA   = {P: t for P, t in load_log_file(weak_path)}

print("\nStrong data loaded:")
for P, t in sorted(STRONG_DATA.items()):
    print(f"  P={P}: total={t.get('total', 0):.1f} ms")
print("\nWeak data loaded:")
for P, t in sorted(WEAK_DATA.items()):
    print(f"  P={P}: total={t.get('total', 0):.1f} ms")

## 2. Strong scaling — speedup and efficiency

In [ ]:
Ps = sorted(STRONG_DATA.keys())
T1 = STRONG_DATA[1]['total']

speedup    = {P: T1 / STRONG_DATA[P]['total'] if STRONG_DATA[P]['total'] > 0 else 0 for P in Ps}
efficiency = {P: speedup[P] / P for P in Ps}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(Ps, [speedup[P] for P in Ps], 'o-', label='Measured')
ax.plot(Ps, Ps, 'k--', label='Ideal')
ax.set_xlabel('Number of ranks P'); ax.set_ylabel('Speedup T(1)/T(P)')
ax.set_title('Strong scaling — N=50M triangles'); ax.legend()
ax.set_xticks(Ps)

ax = axes[1]
ax.plot(Ps, [efficiency[P] * 100 for P in Ps], 's-', color='tab:orange')
ax.axhline(100, color='k', linestyle='--', label='Ideal')
ax.set_xlabel('Number of ranks P'); ax.set_ylabel('Parallel efficiency (%)')
ax.set_title('Strong scaling efficiency'); ax.set_xticks(Ps)
ax.set_ylim(0, 110)

plt.tight_layout()
plt.savefig('../latex/m4_strong_speedup.png', bbox_inches='tight')
plt.show()
print('Speedup:', speedup)
print('Efficiency:', {P: f"{efficiency[P]*100:.1f}%" for P in Ps})

## 3. Weak scaling — normalized wall time

In [ ]:
Ps_w = sorted(WEAK_DATA.keys())
T1_w = WEAK_DATA[1]['total'] if WEAK_DATA[1]['total'] > 0 else 1

norm_time = [WEAK_DATA[P]['total'] / T1_w for P in Ps_w]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(Ps_w, norm_time, 'D-', label='Measured')
ax.axhline(1.0, color='k', linestyle='--', label='Ideal (flat)')
ax.set_xlabel('Number of ranks P'); ax.set_ylabel('Normalized wall time T(P)/T(1)')
ax.set_title('Weak scaling — 10M triangles/rank'); ax.legend()
ax.set_xticks(Ps_w)
plt.tight_layout()
plt.savefig('../latex/m4_weak_norm.png', bbox_inches='tight')
plt.show()

## 4. Per-stage breakdown (compute vs. communication)

In [ ]:
compute_stages = ['A_morton', 'B1_sort', 'C_build']
comm_stages    = ['B2_allgather', 'B3_alltoallv', 'D_merge']

compute_ms = {P: sum(STRONG_DATA[P].get(s, 0) for s in compute_stages) for P in Ps}
comm_ms    = {P: sum(STRONG_DATA[P].get(s, 0) for s in comm_stages)    for P in Ps}

x = np.arange(len(Ps))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, [compute_ms[P] for P in Ps], w, label='Compute')
ax.bar(x + w/2, [comm_ms[P]    for P in Ps], w, label='Communication')
ax.set_xticks(x); ax.set_xticklabels([f'P={P}' for P in Ps])
ax.set_ylabel('Time (ms)'); ax.set_title('Compute vs. Communication (strong scaling)')
ax.legend()
plt.tight_layout()
plt.savefig('../latex/m4_compute_vs_comm_perf.png', bbox_inches='tight')
plt.show()

for P in Ps:
    tot = compute_ms[P] + comm_ms[P]
    if tot > 0:
        print(f'P={P}: compute={compute_ms[P]:.1f}ms ({100*compute_ms[P]/tot:.0f}%)  '
              f'comm={comm_ms[P]:.1f}ms ({100*comm_ms[P]/tot:.0f}%)')

## 5. Isoefficiency analysis

Isoefficiency condition: to maintain efficiency E as P increases, we need

$$W(P) \geq \frac{E}{1-E} \cdot P \cdot T_{\text{comm}}(P)$$

where $W = N \cdot T_{\text{comp\_per\_elem}}$ is the total work.

Alltoallv cost follows $T = t_s + (N/P) \cdot t_w$, so per-rank communication grows sublinearly with N. We report the isoefficiency function $W(P)$ empirically from measured $T_{\text{comm}}/T_{\text{comp}}$ ratios.

In [ ]:
E_target = 0.5  # 50% efficiency target

print(f'Isoefficiency analysis (target efficiency = {E_target*100:.0f}%)')
print(f'{"P":>4}  {"comm_ms":>10}  {"comp_ms":>10}  {"comm/comp":>12}  {"W to maintain 50% eff":>22}')
for P in Ps:
    comp = compute_ms[P]
    comm = comm_ms[P]
    ratio = comm / comp if comp > 0 else float('inf')
    # W(P) >= E/(1-E) * overhead: overhead ≈ P * comm_ms
    # Work W ~ comp_ms; required W = E/(1-E) * P * comm_ms
    required_W = (E_target / (1 - E_target)) * P * comm
    print(f'{P:>4}  {comm:>10.2f}  {comp:>10.2f}  {ratio:>12.3f}  {required_W:>22.1f} ms·ranks')

## 6. Roofline — single-GPU baseline (M3) vs. distributed (M4)

Compare M3 single-GPU numbers with M4 per-rank throughput at P=1,2,4.

Hardware: Quadro RTX 6000 (sm_75)
- Peak HBM bandwidth: 672 GB/s
- Peak FP32 throughput: 16.31 TFLOPS

In [ ]:
PEAK_BW_GBs   = 672.0
PEAK_FP32_GFs = 16310.0
RIDGE_POINT   = PEAK_FP32_GFs / PEAK_BW_GBs

M3_N = 1_000_000
m3_kernels = {
    'Morton (M3)':       {'t_ms': 0.28, 'bytes': M3_N * (36 + 8),       'flops': M3_N * 45},
    'CUB sort (M3)':     {'t_ms': 0.45, 'bytes': M3_N * 32 * 4,         'flops': M3_N * 0.4},
    'Karras (M3)':       {'t_ms': 3.50, 'bytes': M3_N * (4 + 12),       'flops': M3_N * 29},
    'Refit atomic (M3)': {'t_ms': 0.46, 'bytes': M3_N * (24 + 12 + 12), 'flops': M3_N * 0.7},
}

fig, ax = plt.subplots(figsize=(9, 5))
I_range = np.logspace(-3, 2, 300)
perf = np.minimum(PEAK_FP32_GFs, PEAK_BW_GBs * I_range)
ax.loglog(I_range, perf, 'k-', lw=2, label='Roofline (1 GPU)')
ax.axvline(RIDGE_POINT, color='gray', linestyle=':', alpha=0.6)

markers = ['o', 's', '^', 'D']
for (name, kd), mk in zip(m3_kernels.items(), markers):
    I    = kd['flops'] / kd['bytes']
    perf_val = kd['flops'] / (kd['t_ms'] * 1e-3) / 1e9
    ax.scatter(I, perf_val, marker=mk, s=80, label=name)

ax.set_xlabel('Arithmetic intensity (FLOP/byte)')
ax.set_ylabel('Performance (GFLOPS)')
ax.set_title('Roofline — M3 single-GPU kernels (Quadro RTX 6000)')
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(1e-3, 1e2); ax.set_ylim(1e0, PEAK_FP32_GFs * 1.5)
plt.tight_layout()
plt.savefig('../latex/m4_roofline.png', bbox_inches='tight')
plt.show()

print(f'Ridge point: {RIDGE_POINT:.1f} FLOP/byte')
print('All kernels are memory-bandwidth bound (I << ridge point)')

## 7. Summary table

In [ ]:
print('=== Strong scaling summary (N=50M triangles) ===')
print(f'{"P":>4}  {"Total(ms)":>10}  {"Speedup":>8}  {"Efficiency":>12}')
for P in Ps:
    t = STRONG_DATA[P]['total']
    sp = speedup[P]
    eff = efficiency[P]
    print(f'{P:>4}  {t:>10.1f}  {sp:>8.2f}  {eff*100:>11.1f}%')

print()
print('=== Weak scaling summary (10M tris/rank) ===')
print(f'{"P":>4}  {"Total(ms)":>10}  {"Norm. time":>12}')
for P in Ps_w:
    t = WEAK_DATA[P]['total']
    print(f'{P:>4}  {t:>10.1f}  {t/T1_w:>12.3f}')